In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from itertools import combinations
import json
import os

In [2]:
# ── CHANGE THESE TWO LINES per run ──────────────────────────────
MODEL_FOLDER = "bert-base-multilingual-cased" # "bert-base-multilingual-cased" or "muril-base-cased" or "xlm-roberta-base"
STATE        = "finetuned" # "pretrained" or "finetuned"
# ────────────────────────────────────────────────────────────────

BASE_DIR   = "/Users/harshaggarwal/Projects_4/hinemo_project/models/hidden_states"
MODEL_DIR  = f"{BASE_DIR}/{MODEL_FOLDER}"
OUTPUT_DIR = f"/Users/harshaggarwal/Projects_4/hinemo_project/models/probing_results/{MODEL_FOLDER}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

EMOTIONS = ["anger", "disgust", "joy", "sadness"]

print(f"Model:  {MODEL_FOLDER}")
print(f"State:  {STATE}")

Model:  bert-base-multilingual-cased
State:  finetuned


In [3]:
hidden      = np.load(f"{MODEL_DIR}/hidden_{STATE}.npy")
meta        = pd.read_csv(f"{MODEL_DIR}/metadata.csv")
lambda_vals = meta["lambda"].values

n_samples, n_layers, hidden_dim = hidden.shape

print(f"Hidden states: {hidden.shape}")
print(f"\nSamples per emotion:")
print(meta["gpt_emotion"].value_counts())

Hidden states: (3000, 12, 768)

Samples per emotion:
gpt_emotion
anger      750
disgust    750
joy        750
sadness    750
Name: count, dtype: int64


In [4]:
r2_per_emotion = {}   # stores full R² curve per emotion
lsl_per_emotion = {}  # stores peak layer per emotion

for emotion in EMOTIONS:
    mask        = meta["gpt_emotion"] == emotion
    X_emotion   = hidden[mask]          # shape: (~750, 12, 768)
    y_emotion   = lambda_vals[mask]     # shape: (~750,)

    print(f"\n── {emotion.upper()} ({mask.sum()} samples) ──")

    r2_curve = []
    for layer in range(n_layers):
        X      = X_emotion[:, layer, :]
        probe  = Ridge(alpha=1.0)
        scores = cross_val_score(probe, X, y_emotion, cv=5, scoring="r2")
        r2_curve.append(scores.mean())
        print(f"  Layer {layer+1:2d}: R² = {scores.mean():.4f}")

    lsl_e = int(np.argmax(r2_curve)) + 1
    r2_per_emotion[emotion]  = r2_curve
    lsl_per_emotion[emotion] = lsl_e

    print(f"  LSL({emotion}) = Layer {lsl_e}  |  Peak R² = {max(r2_curve):.4f}")


── ANGER (750 samples) ──
  Layer  1: R² = 0.5624
  Layer  2: R² = 0.4910
  Layer  3: R² = 0.4989
  Layer  4: R² = 0.6105
  Layer  5: R² = 0.4024
  Layer  6: R² = -0.1246
  Layer  7: R² = -0.2682
  Layer  8: R² = -0.4386
  Layer  9: R² = -0.3019
  Layer 10: R² = -0.6075
  Layer 11: R² = -0.8022
  Layer 12: R² = -0.2271
  LSL(anger) = Layer 4  |  Peak R² = 0.6105

── DISGUST (750 samples) ──
  Layer  1: R² = 0.5639
  Layer  2: R² = 0.4990
  Layer  3: R² = 0.5078
  Layer  4: R² = 0.5937
  Layer  5: R² = 0.4708
  Layer  6: R² = 0.0037
  Layer  7: R² = -0.3146
  Layer  8: R² = -0.3222
  Layer  9: R² = -0.3911
  Layer 10: R² = -0.6807
  Layer 11: R² = -0.7163
  Layer 12: R² = -0.2067
  LSL(disgust) = Layer 4  |  Peak R² = 0.5937

── JOY (750 samples) ──
  Layer  1: R² = 0.6573
  Layer  2: R² = 0.6239
  Layer  3: R² = 0.6281
  Layer  4: R² = 0.6776
  Layer  5: R² = 0.6374
  Layer  6: R² = 0.4237
  Layer  7: R² = 0.0997
  Layer  8: R² = 0.0863
  Layer  9: R² = -0.1143
  Layer 10: R² = -0.278

In [5]:
print("\n── EMOTION LSL SUMMARY ──")
print(f"{'Emotion':<10} {'LSL':<6} {'Peak R²'}")
print("-" * 30)

for emotion in EMOTIONS:
    lsl  = lsl_per_emotion[emotion]
    peak = max(r2_per_emotion[emotion])
    print(f"{emotion:<10} {lsl:<6} {peak:.4f}")


── EMOTION LSL SUMMARY ──
Emotion    LSL    Peak R²
------------------------------
anger      4      0.6105
disgust    4      0.5937
joy        4      0.6776
sadness    4      0.7201


In [6]:
pairs         = list(combinations(EMOTIONS, 2))
pairwise_diffs = [abs(lsl_per_emotion[e1] - lsl_per_emotion[e2]) for e1, e2 in pairs]
ELDS          = sum(pairwise_diffs) / len(pairwise_diffs)

print("── PAIRWISE LSL DIFFERENCES ──")
for (e1, e2), diff in zip(pairs, pairwise_diffs):
    print(f"  |LSL({e1}) - LSL({e2})| = {diff}")

print(f"\nELDS = {ELDS:.4f}")

── PAIRWISE LSL DIFFERENCES ──
  |LSL(anger) - LSL(disgust)| = 0
  |LSL(anger) - LSL(joy)| = 0
  |LSL(anger) - LSL(sadness)| = 0
  |LSL(disgust) - LSL(joy)| = 0
  |LSL(disgust) - LSL(sadness)| = 0
  |LSL(joy) - LSL(sadness)| = 0

ELDS = 0.0000


In [8]:
results = {
    "model"           : MODEL_FOLDER,
    "state"           : STATE,
    "lsl_per_emotion" : lsl_per_emotion,
    "r2_per_emotion"  : r2_per_emotion,
    "ELDS"            : ELDS,
    "pairwise_diffs"  : {f"{e1}_vs_{e2}": abs(lsl_per_emotion[e1] - lsl_per_emotion[e2]) 
                         for e1, e2 in pairs}
}

save_path = f"{OUTPUT_DIR}/emotion_probing_{STATE}.json"
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved to {save_path}")

Saved to /Users/harshaggarwal/Projects_4/hinemo_project/models/probing_results/bert-base-multilingual-cased/emotion_probing_finetuned.json


In [7]:
n_permutations  = 1000
permuted_elds   = []

for perm in range(n_permutations):
    # Shuffle emotion labels randomly
    shuffled = meta["gpt_emotion"].sample(frac=1, random_state=perm).values

    perm_lsl = {}
    for emotion in EMOTIONS:
        mask    = shuffled == emotion
        X_perm  = hidden[mask]
        y_perm  = lambda_vals[mask]

        r2s = []
        for layer in range(n_layers):
            X      = X_perm[:, layer, :]
            probe  = Ridge(alpha=1.0)
            scores = cross_val_score(probe, X, y_perm, cv=5, scoring="r2")
            r2s.append(scores.mean())

        perm_lsl[emotion] = int(np.argmax(r2s)) + 1

    diffs = [abs(perm_lsl[e1] - perm_lsl[e2]) for e1, e2 in pairs]
    permuted_elds.append(sum(diffs) / len(diffs))

    if (perm + 1) % 100 == 0:
        print(f"  {perm+1}/1000 done...")

permuted_array = np.array(permuted_elds)
p_value        = (permuted_array >= ELDS).mean()

print(f"\nReal ELDS:        {ELDS:.4f}")
print(f"Mean permuted:    {permuted_array.mean():.4f}")
print(f"p-value:          {p_value:.4f}")
print(f"Significant p<0.05: {p_value < 0.05}")

  100/1000 done...
  200/1000 done...


KeyboardInterrupt: 